# 🏆 ระบบตรวจ Final — Agentic RAG Workshop
## รวมคะแนน 3 วัน × 3 ด้าน

---
### วิธีใช้
1. Run Setup → อัปโหลด notebook ของนักศึกษา
2. ตรวจทีละคน → AI ให้คะแนน 3 ด้าน (Responsibility / Determination / Technical)
3. ดูสรุป + Export CSV

## 📦 Setup (Run ครั้งเดียว)

In [ ]:
%%time
!pip install -q google-genai nbformat

import os, json, csv, re, hashlib, random
from datetime import datetime
import nbformat
from google import genai

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    GOOGLE_API_KEY = input('🔑 Gemini API Key: ')

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-2.5-pro'

FINAL_CSV = 'final_scores.csv'
FINAL_JSON = 'final_scores.json'

CSV_HEADER = ['Timestamp','Name','Student ID','Phone','LINE','Day',
    'Responsibility','Determination','Technical','Day Total',
    'R_feedback','D_feedback','T_feedback','AI Suspected','AI Reason']

if not os.path.exists(FINAL_CSV):
    with open(FINAL_CSV, 'w', encoding='utf-8', newline='') as f:
        csv.writer(f).writerow(CSV_HEADER)
if not os.path.exists(FINAL_JSON):
    with open(FINAL_JSON, 'w') as f: json.dump([], f)

print(f'✅ Setup | Model: {MODEL}')

## 🔧 ฟังก์ชันตรวจ (Run ครั้งเดียว)

In [ ]:
def read_notebook(path):
    """อ่าน notebook จาก file path"""
    with open(path) as f:
        nb = nbformat.read(f, as_version=4)
    info = {'student_name':'','student_id':'','phone':'','line_id':''}
    content = ''
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'code':
            for key, var in [('student_id','STUDENT_ID'),('student_name','STUDENT_NAME'),
                             ('phone','PHONE'),('line_id','LINE_ID')]:
                m = re.search(rf"{var}\s*=\s*['\"](.*?)['\"]", cell.source)
                if m: info[key] = m.group(1)
        content += f'\n=== CELL {i} ({cell.cell_type.upper()}) ===\n{cell.source}'
        if hasattr(cell, 'outputs'):
            for out in cell.outputs:
                if hasattr(out, 'text'): content += f'\n--- OUTPUT ---\n{out.text}'
                elif hasattr(out, 'data'):
                    for mime, data in out.data.items():
                        if 'text' in mime:
                            content += f'\n--- OUTPUT ---\n{data if isinstance(data,str) else chr(10).join(data)}'
    return info, content


def grade_3d(content, day_num, deadline_str=''):
    """ตรวจ 3 ด้าน: Responsibility, Determination, Technical"""
    prompt = f'''ตรวจการบ้าน Day {day_num} ด้วยเกณฑ์ 3 ด้าน:\n\n
Notebook:\n{content[:15000]}\n\n
=== เกณฑ์ ===\n
1. Responsibility (10): ส่งตรงเวลา(3) กรอกข้อมูลครบ(2) Run ตรวจสอบก่อนส่ง(2) ส่งครบ(3)\n
   - ตรวจว่ามี STUDENT_NAME, STUDENT_ID ครบ → 2 คะแนน\n
   - ตรวจว่ามี output จาก validation cell (✅ตรวจสอบ) → 2 คะแนน\n
   - {f"deadline: {deadline_str}" if deadline_str else "ไม่ระบุ deadline → ให้ 3 เต็ม"}\n
   - ส่งมาแล้ว → 3 เต็ม (ตรวจจำนวน Day ทีหลัง)\n\n
2. Determination (10): ทำครบทุกข้อ(3) ลองปรับ parameter(3) อธิบาย/วิเคราะห์(2) แก้ปัญหา(2)\n
   - ถ้ามีแค่ starter code ไม่ได้เขียนเพิ่ม → 0\n
   - ถ้าลอง parameter หลายค่า มี comment → เต็ม\n\n
3. Technical (10): code ทำงาน(3) ผลถูกต้อง(3) คุณภาพ code(2) ไม่ copy/AI(2)\n
   - ดูจาก output ว่ามี error มั้ย\n
   - ดูว่าผลตรงกับข้อมูลเฉพาะตัว (STUDENT_ID)\n
   - ถ้าสงสัยว่าใช้ AI ทำทั้งหมด → is_ai_suspected=true\n\n
ตอบ JSON:\n
{{"responsibility_score":0,"responsibility_feedback":"...",
"determination_score":0,"determination_feedback":"...",
"technical_score":0,"technical_feedback":"...",
"day_total":0,"overall_feedback":"...",
"is_ai_suspected":false,"ai_reason":"..."}}
feedback ภาษาไทย คะแนนแต่ละด้าน max 10 total max 30'''
    resp = client.models.generate_content(model=MODEL, contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
    return json.loads(resp.text)


def append_final(info, grade, day_num):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(FINAL_CSV, 'a', encoding='utf-8', newline='') as f:
        csv.writer(f).writerow([
            ts, info['student_name'], info['student_id'], info['phone'], info['line_id'],
            f'Day{day_num}',
            grade['responsibility_score'], grade['determination_score'], grade['technical_score'],
            grade['day_total'],
            grade['responsibility_feedback'], grade['determination_feedback'], grade['technical_feedback'],
            grade.get('is_ai_suspected', False), grade.get('ai_reason', '')
        ])
    with open(FINAL_JSON, 'r') as f: all_r = json.load(f)
    grade.update({'timestamp': ts, 'day': day_num, **info})
    all_r.append(grade)
    with open(FINAL_JSON, 'w', encoding='utf-8') as f: json.dump(all_r, f, ensure_ascii=False, indent=2)

print('✅ ฟังก์ชันพร้อม')

---
## ▶️ ตรวจการบ้าน (ทีละคน ทีละ Day)

In [ ]:
# ============================================================
# 👇 ตั้งค่า
# ============================================================
NOTEBOOK_PATH = ''    # path ไปยังไฟล์ .ipynb ที่อัปโหลด
DAY_NUM = 1           # 1, 2, หรือ 3
DEADLINE = ''         # เช่น '2026-03-12' (ถ้าไม่ระบุ = ไม่หักคะแนน)
# ============================================================

assert NOTEBOOK_PATH, '❌ กรุณาระบุ path ของ notebook!'
assert DAY_NUM in [1,2,3], '❌ DAY_NUM ต้องเป็น 1, 2 หรือ 3'

info, content = read_notebook(NOTEBOOK_PATH)
print(f'👤 {info["student_name"]} ({info["student_id"]})')

# ตรวจ duplicate
already = False
if os.path.exists(FINAL_JSON):
    with open(FINAL_JSON) as f:
        for p in json.load(f):
            if p.get('student_id')==info['student_id'] and p.get('day')==DAY_NUM:
                already=True
                print(f'⚠️ เคยตรวจ Day {DAY_NUM} แล้ว (R:{p.get("responsibility_score")} D:{p.get("determination_score")} T:{p.get("technical_score")} = {p.get("day_total")})')
                break
if already:
    ow = input('🔄 ตรวจใหม่? (y/n): ').strip().lower()
    if ow != 'y': raise SystemExit('⏭️ ข้าม')
    with open(FINAL_JSON) as f: prev=json.load(f)
    prev=[p for p in prev if not (p.get('student_id')==info['student_id'] and p.get('day')==DAY_NUM)]
    with open(FINAL_JSON,'w',encoding='utf-8') as f: json.dump(prev,f,ensure_ascii=False,indent=2)
    rows=[]
    with open(FINAL_CSV,'r',encoding='utf-8') as f: rows=list(csv.reader(f))
    with open(FINAL_CSV,'w',encoding='utf-8',newline='') as f:
        w=csv.writer(f)
        for r in rows:
            if len(r)>2 and r[2]==info['student_id'] and r[5]==f'Day{DAY_NUM}': continue
            w.writerow(r)

print(f'🤖 ตรวจ Day {DAY_NUM} ด้วย {MODEL}...')
grade = grade_3d(content, DAY_NUM, DEADLINE)

print(f'\n{"═"*60}')
print(f'📊 ผลการตรวจ Day {DAY_NUM}: {info["student_name"]} ({info["student_id"]})')
print(f'{"═"*60}')
print(f'  🤝 Responsibility:  {grade["responsibility_score"]:>5}/10  — {grade["responsibility_feedback"]}')
print(f'  💪 Determination:   {grade["determination_score"]:>5}/10  — {grade["determination_feedback"]}')
print(f'  💻 Technical:       {grade["technical_score"]:>5}/10  — {grade["technical_feedback"]}')
print(f'  {"─"*56}')
print(f'  🏆 Day {DAY_NUM} Total:    {grade["day_total"]:>5}/30')
if grade.get('is_ai_suspected'): print(f'  ⚠️ AI Suspected: {grade["ai_reason"]}')

append_final(info, grade, DAY_NUM)
print(f'\n💾 บันทึกแล้ว')

## 📊 สรุปคะแนนรวม (ทุกคน ทุก Day)

In [ ]:
with open(FINAL_JSON, 'r') as f:
    all_results = json.load(f)

# Group by student
students = {}
for r in all_results:
    sid = r.get('student_id', '?')
    if sid not in students:
        students[sid] = {'name': r.get('student_name',''), 'days': {}}
    students[sid]['days'][r.get('day', 0)] = r

print(f'📊 นักศึกษา {len(students)} คน\n')
print(f'{"#":>3} {"ชื่อ":<18} {"รหัส":<12} {"D1":>4} {"D2":>4} {"D3":>4} {"R":>4} {"D":>4} {"T":>4} {"Total":>6} {"AI?":>4}')
print('═' * 85)

totals = []
for i, (sid, s) in enumerate(sorted(students.items()), 1):
    d1 = s['days'].get(1, {})
    d2 = s['days'].get(2, {})
    d3 = s['days'].get(3, {})
    
    r_total = sum(d.get('responsibility_score', 0) for d in [d1, d2, d3])
    d_total = sum(d.get('determination_score', 0) for d in [d1, d2, d3])
    t_total = sum(d.get('technical_score', 0) for d in [d1, d2, d3])
    grand = r_total + d_total + t_total
    
    day1_t = d1.get('day_total', '-')
    day2_t = d2.get('day_total', '-')
    day3_t = d3.get('day_total', '-')
    
    ai = '⚠️' if any(d.get('is_ai_suspected', False) for d in [d1, d2, d3]) else '✅'
    
    print(f'{i:>3} {s["name"]:<18} {sid:<12} {str(day1_t):>4} {str(day2_t):>4} {str(day3_t):>4} {r_total:>4} {d_total:>4} {t_total:>4} {grand:>6}/90 {ai:>4}')
    totals.append(grand)

if totals:
    print(f'\n📈 min={min(totals):.0f}, max={max(totals):.0f}, avg={sum(totals)/len(totals):.1f}')

## 📋 รายละเอียดรายคน

In [ ]:
# ดูรายละเอียดคนใดคนหนึ่ง
VIEW_STUDENT_ID = ''  # ใส่รหัส

if VIEW_STUDENT_ID and VIEW_STUDENT_ID in students:
    s = students[VIEW_STUDENT_ID]
    print(f'👤 {s["name"]} ({VIEW_STUDENT_ID})')
    print(f'{"═"*60}')
    for d in [1, 2, 3]:
        if d in s['days']:
            r = s['days'][d]
            print(f'\n📋 Day {d}:')
            print(f'  🤝 R: {r.get("responsibility_score",0)}/10 — {r.get("responsibility_feedback","")}')
            print(f'  💪 D: {r.get("determination_score",0)}/10 — {r.get("determination_feedback","")}')
            print(f'  💻 T: {r.get("technical_score",0)}/10 — {r.get("technical_feedback","")}')
            if r.get('is_ai_suspected'): print(f'  ⚠️ AI: {r.get("ai_reason","")}')
        else:
            print(f'\n📋 Day {d}: ❌ ยังไม่ได้ส่ง')
else:
    print('กรุณาระบุ VIEW_STUDENT_ID')

## 💾 ดาวน์โหลด

In [ ]:
try:
    from google.colab import files
    files.download(FINAL_CSV)
except:
    print(f'📄 CSV: {os.path.abspath(FINAL_CSV)}')
    print(f'📄 JSON: {os.path.abspath(FINAL_JSON)}')